In [2]:
# 1
import numpy as np
# 2
import pandas as pd
# 3
from sklearn.datasets import load_iris
# 4
from sklearn.model_selection import train_test_split, cross_val_score
# 5
from sklearn.preprocessing import StandardScaler
# 6
from sklearn.neighbors import KNeighborsClassifier
# 7
from sklearn.svm import SVC
# 8
from sklearn.linear_model import LogisticRegression
# 9
from sklearn.tree import DecisionTreeClassifier
# 10
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# 11
import joblib

# 12
iris = load_iris()
# 13
X = iris.data
# 14
y = iris.target
# 15
feature_names = iris.feature_names
# 16
target_names = iris.target_names

# 17
df = pd.DataFrame(X, columns=feature_names)
# 18
df['target'] = y

# 19
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 20
scaler = StandardScaler()
# 21
X_train = scaler.fit_transform(X_train)
# 22
X_test = scaler.transform(X_test)

# 23
models = {
    'LogisticRegression': LogisticRegression(max_iter=200, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'DecisionTree': DecisionTreeClassifier(random_state=42)
}

# 24
results = {}
# 25
for name, model in models.items():
    # 26
    model.fit(X_train, y_train)
    # 27
    y_pred = model.predict(X_test)
    # 28
    acc = accuracy_score(y_test, y_pred)
    # 29
    results[name] = {
        'accuracy': acc,
        'report': classification_report(y_test, y_pred, target_names=target_names, zero_division=0),
        'confusion': confusion_matrix(y_test, y_pred)
    }
    # 30
    print(f"---{name}---")
    # 31
    print("Accuracy:", acc)
    # 32
    print(results[name]['report'])
    # 33
    print("Confusion matrix:\n", results[name]['confusion'])
    # 34
    print()

# 35
X_scaled = scaler.fit_transform(X)
# 36
for name, model in models.items():
    # 37
    scores = cross_val_score(model, X_scaled, y, cv=5)
    # 38
    print(f"{name} 5-fold CV accuracy: mean={scores.mean():.3f}, std={scores.std():.3f}")

# 39
best_name = max(results, key=lambda k: results[k]['accuracy'])
# 40
best_model = models[best_name]
# 41
joblib.dump({
    'model': best_model,
    'scaler': scaler,
    'feature_names': feature_names,
    'target_names': target_names
}, 'iris_best_model.pkl')
# 42
print(f"\nSaved best model: {best_name} to iris_best_model.pkl")


---LogisticRegression---
Accuracy: 0.9333333333333333
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

Confusion matrix:
 [[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]

---KNN---
Accuracy: 0.9333333333333333
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.83      1.00      0.91        10
   virginica       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30

Confusion matrix:
 [[10  0  0]
 [ 0 10  0]
 [ 0  2  8]]

---SVM---
Accuracy: 0.966666666